# Disease Prediction from Medical Data Using Machine Learning

## Loading the Datasets

In [2]:

import pandas as pd

diabetes = pd.read_csv("diabetes.csv")
heart = pd.read_csv("heart.csv")

print("Diabetes Columns:")
print(diabetes.columns)

print("Heart Disease Columns:")
print(heart.columns)

Diabetes Columns:
Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')
Heart Disease Columns:
Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'],
      dtype='object')


## Data Preprocessing and Data Cleaning

In [3]:
print("Diabetes:", diabetes.shape)
print("Heart:", heart.shape)

Diabetes: (768, 9)
Heart: (1025, 14)


In [4]:
print(diabetes.head())
print(heart.head())

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      

In [5]:
print(diabetes["Outcome"].value_counts())
print(heart["target"].value_counts())

Outcome
0    500
1    268
Name: count, dtype: int64
target
1    526
0    499
Name: count, dtype: int64


In [6]:
print(diabetes.isnull().sum())

print(heart.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


In [7]:
print("Diabetes duplicates:", diabetes.duplicated().sum())
print("Heart duplicates:", heart.duplicated().sum())

Diabetes duplicates: 0
Heart duplicates: 723


In [8]:
heart = heart.drop_duplicates()


In [9]:
print("Heart duplicates:", heart.duplicated().sum())

Heart duplicates: 0


In [10]:
diabetes['Disease'] = 0
heart['Disease'] = 1


In [11]:
diabetes_new = pd.DataFrame({
    'Age': diabetes['Age'],
    'BloodPressure': diabetes['BloodPressure'],
    'BMI': diabetes['BMI'],
    'Glucose': diabetes['Glucose'],
    'Disease': diabetes['Disease']
})

In [12]:
heart_new = pd.DataFrame({
    'Age': heart['age'],
    'BloodPressure': heart['trestbps'],
    'BMI': diabetes['BMI'].mean(),
    'Glucose': diabetes['Glucose'].mean(),
    'Disease': heart['Disease']
})

## Dataset Merging

In [13]:
merged_df = pd.concat(
    [diabetes_new, heart_new],
    ignore_index=True
)

In [14]:
print(merged_df.shape)

print(merged_df.head())

print(merged_df['Disease'].value_counts())

(1070, 5)
   Age  BloodPressure   BMI  Glucose  Disease
0   50             72  33.6    148.0        0
1   31             66  26.6     85.0        0
2   32             64  23.3    183.0        0
3   21             66  28.1     89.0        0
4   33             40  43.1    137.0        0
Disease
0    768
1    302
Name: count, dtype: int64


In [15]:
merged_df.to_csv(
    "multi_disease_dataset.csv",
    index=False
)

## Feature and Target Selection

In [16]:
X = merged_df.drop('Disease', axis=1)
y = merged_df['Disease']

## Data Splitting and Feature Scaling

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Logistic Regression Model

In [19]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [20]:
y_pred_lr = lr_model.predict(X_test)

In [21]:
from sklearn.metrics import accuracy_score

lr_accuracy = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", lr_accuracy)

Logistic Regression Accuracy: 0.985981308411215


## Support Vector Machine (SVM)

In [22]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

svm_model = SVC()

svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("SVM Accuracy:", svm_accuracy)

SVM Accuracy: 0.9953271028037384


## Random Forest Model

In [23]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", rf_accuracy)

Random Forest Accuracy: 1.0


## XGBoost Model

In [24]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

print("XGBoost Accuracy:", xgb_accuracy)

XGBoost Accuracy: 0.9953271028037384


D:\python\Lib\site-packages\xgboost\training.py:199: UserWarning: [08:45:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## Model Performance Comparison

In [25]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        lr_accuracy,
        svm_accuracy,
        rf_accuracy,
        xgb_accuracy
    ]
})

print(results)

                 Model  Accuracy
0  Logistic Regression  0.985981
1                  SVM  0.995327
2        Random Forest  1.000000
3              XGBoost  0.995327


In [27]:
from sklearn.metrics import accuracy_score
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM",
        "Random Forest",
        "XGBoost"
    ],
    "Training Accuracy": [
        accuracy_score(y_train, lr_model.predict(X_train)),
        accuracy_score(y_train, svm_model.predict(X_train)),
        accuracy_score(y_train, rf_model.predict(X_train)),
        accuracy_score(y_train, xgb_model.predict(X_train))
    ],
    "Testing Accuracy": [
        accuracy_score(y_test, lr_model.predict(X_test)),
        accuracy_score(y_test, svm_model.predict(X_test)),
        accuracy_score(y_test, rf_model.predict(X_test)),
        accuracy_score(y_test, xgb_model.predict(X_test))
    ]
})

print(comparison)

                 Model  Training Accuracy  Testing Accuracy
0  Logistic Regression           0.988318          0.985981
1                  SVM           0.997664          0.995327
2        Random Forest           1.000000          1.000000
3              XGBoost           1.000000          0.995327


## Cross Validation

In [28]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf_model, X, y, cv=5)

print("Cross Validation Scores:", scores)
print("Average Accuracy:", scores.mean())

Cross Validation Scores: [0.9953271  1.         0.99065421 0.9953271  1.        ]
Average Accuracy: 0.9962616822429908
